In [3]:
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import openpyxl
import os
from collections import Counter

def ensure_package(package_name):
    try:
        __import__(package_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

ensure_package("numpy")
ensure_package("matplotlib")

In [4]:
NIST_DESCRIPTORS = {
    # (descriptor_string): (include, intensity_multiplier, width_multiplier, confidence_label, visual_action)
    '*': (True, 1.0, 1.0, 'medium', 'line'),          # Shared intensity
    ':': (True, 1.0, 1.0, 'high', 'line'),            # Rounded Ritz value
    '-': (True, 0.8, 1.0, 'medium', 'line'),          # Somewhat lower intensity
    'a': (True, 0.5, 1.0, 'low', 'line'),             # Absorption
    'b': (True, 1.0, 1.2, 'high', 'band-edge'),       # Band head
    'bl': (True, 1.0, 1.25, 'medium', 'blend'),       # Blended
    'B': (True, 1.0, 1.8, 'high', 'broad'),           # Autoionization broadening
    'c': (True, 1.0, 1.1, 'high', 'line'),            # Complex
    'd': (True, 1.0, 1.1, 'high', 'line'),            # Diffuse
    'D': (True, 1.0, 1.0, 'high', 'doublet'),         # Double line; two nearby components, not a broader single line
    'E': (True, 0.9, 1.3, 'medium', 'broad'),         # Overexposed
    'f': (True, 1.0, 1.0, 'high', 'line'),            # Forbidden
    'g': (True, 1.0, 1.0, 'high', 'line'),            # Ground term
    'G': (True, 1.0, 1.0, 'low', 'line'),             # Roughly estimated wavelength
    'H': (True, 0.7, 1.5, 'low', 'broad'),            # Very hazy
    'h': (True, 1.0, 1.1, 'high', 'line'),            # Hazy (diffuse)
    'hfs': (True, 1.0, 1.35, 'high', 'cluster'),      # Hyperfine structure
    'i': (True, 0.6, 1.0, 'low', 'line'),             # Uncertain identification
    'j': (True, 1.0, 1.0, 'high', 'line'),            # Wavelength smoothed
    'l': (True, 1.0, 1.0, 'high', 'line'),            # Shaded to longer wavelengths
    'm': (False, 0.0, 0.0, 'excluded', 'skip'),       # Masked — skip
    'p': (True, 0.8, 1.2, 'medium', 'line'),          # Perturbed
    'q': (True, 1.0, 1.0, 'high', 'line'),            # Asymmetric
    'r': (True, 1.0, 1.0, 'high', 'line'),            # Easily reversed
    's': (True, 1.0, 1.0, 'high', 'line'),            # Shaded to shorter wavelengths
    't': (True, 0.7, 1.0, 'low', 'line'),             # Tentative
    'u': (True, 0.75, 1.2, 'medium', 'cluster'),      # Unresolved
    'w': (True, 1.0, 1.8, 'high', 'broad'),           # Wide
    'x': (True, 1.0, 1.0, 'low', 'line'),             # Extrapolated wavelength
}

DESCRIPTOR_KEYS = sorted(NIST_DESCRIPTORS.keys(), key=len, reverse=True)
CONFIDENCE_ORDER = {'high': 0, 'medium': 1, 'low': 2, 'excluded': 3}

In [8]:
def clean_title(text):
    text = str(text).strip()
    if not text:
        return None
    return text.split()[0]

# *** #

def normalize_header(value):
    return str(value).strip().lower() if value is not None else ''

def looks_like_number(value):
    try:
        float(value)
        return True
    except (TypeError, ValueError):
        return False

def split_descriptor_tokens(descriptor_text):
    """Split composite descriptor strings like 'bl*' or 'w*' into known tokens."""
    remaining = descriptor_text.strip()
    tokens = []
    while remaining:
        matched = None
        for key in DESCRIPTOR_KEYS:
            if remaining.startswith(key):
                matched = key
                break
        if matched is None:
            # Consume one unknown character to avoid infinite loop
            tokens.append(remaining[0])
            remaining = remaining[1:]
            continue
        tokens.append(matched)
        remaining = remaining[len(matched):]
    return tokens

def combine_descriptor_rules(tokens):
    include = True
    intensity_multiplier = 1.0
    width_multiplier = 1.0
    confidence = 'high'
    actions = []
    unknown_tokens = []

    for token in tokens:
        rule = NIST_DESCRIPTORS.get(token)
        if rule is None:
            unknown_tokens.append(token)
            continue
        token_include, token_intensity, token_width, token_confidence, token_action = rule
        include = include and token_include
        intensity_multiplier *= token_intensity
        width_multiplier = max(width_multiplier, token_width)
        if CONFIDENCE_ORDER[token_confidence] > CONFIDENCE_ORDER[confidence]:
            confidence = token_confidence
        if token_action not in actions:
            actions.append(token_action)

    return include, intensity_multiplier, width_multiplier, confidence, actions, unknown_tokens

def parse_nist_intensity(intensity_value):
    """Parse numeric intensity plus descriptor suffix. Example: '100bl*'."""
    if intensity_value is None:
        return None, '', [], 1.0, 1.0, 'high', [], []

    value_str = str(intensity_value).strip()
    i = 0
    while i < len(value_str) and (value_str[i].isdigit() or value_str[i] in '.+-'):
        i += 1

    numeric_part = value_str[:i].strip()
    descriptor_text = value_str[i:].strip()

    if not numeric_part:
        return None, descriptor_text, [], 1.0, 1.0, 'high', [], []

    try:
        numeric_intensity = float(numeric_part)
    except ValueError:
        return None, descriptor_text, [], 1.0, 1.0, 'high', [], []

    tokens = split_descriptor_tokens(descriptor_text) if descriptor_text else []
    include, intensity_multiplier, width_multiplier, confidence, actions, unknown_tokens = combine_descriptor_rules(tokens)

    return numeric_intensity, descriptor_text, tokens, intensity_multiplier, width_multiplier, confidence, actions, unknown_tokens

def find_spectral_sheet(workbook):
    wavelength_tokens = ('wavelength', 'wave', 'lambda', 'nm')
    intensity_tokens = ('intensity', 'rel int', 'relative intensity', 'rel. int', 'rel', 'int', 'counts', 'count', 'signal', 'flux', 'value')
    title_tokens = ('spectrum', 'element', 'species', 'ion', 'atom', 'source')
    for sheet in workbook.worksheets:
        for header_row in range(1, min(sheet.max_row, 10) + 1):
            headers = [normalize_header(cell.value) for cell in sheet[header_row]]
            wavelength_col = next((i for i, header in enumerate(headers) if any(token in header for token in wavelength_tokens)), None)
            intensity_col = next((i for i, header in enumerate(headers) if any(token in header for token in intensity_tokens)), None)
            if wavelength_col is None or intensity_col is None:
                continue
            title_col = next((i for i, header in enumerate(headers) if any(token in header for token in title_tokens)), None)
            return sheet, header_row, wavelength_col, intensity_col, title_col
    return None, None, None, None, None

ws, header_row, wavelength_col, intensity_col, title_col = find_spectral_sheet(wb)
if ws is None:
    raise ValueError('No sheet with recognizable wavelength and intensity columns was found')

wavelengths = []
intensities = []
line_width_factors = []
line_confidences = []
line_actions = []
title_candidates = []
raw_descriptor_counts = Counter()
parsed_descriptor_counts = Counter()
unknown_descriptor_counts = Counter()
skipped_masked_lines = 0

for row in ws.iter_rows(min_row=header_row + 1, values_only=True):
    if row is None:
        continue
    if wavelength_col >= len(row) or intensity_col >= len(row):
        continue

    wavelength_value = row[wavelength_col]
    intensity_value = row[intensity_col]
    if wavelength_value is None or intensity_value is None:
        continue

    try:
        wl = float(wavelength_value)
    except (ValueError, TypeError):
        continue

    numeric_intensity, raw_descriptor, tokens, intensity_multiplier, width_multiplier, confidence, actions, unknown_tokens = parse_nist_intensity(intensity_value)
    if numeric_intensity is None:
        continue

    if raw_descriptor:
        raw_descriptor_counts[raw_descriptor] += 1
    for token in tokens:
        parsed_descriptor_counts[token] += 1
    for token in unknown_tokens:
        unknown_descriptor_counts[token] += 1

    # Skip masked lines (`m`) or any descriptor that maps to exclusion
    include_line = 'm' not in tokens
    if not include_line:
        skipped_masked_lines += 1
        continue

    corrected_intensity = numeric_intensity * intensity_multiplier

    wavelengths.append(wl)
    intensities.append(corrected_intensity)
    line_width_factors.append(width_multiplier)
    line_confidences.append(confidence)
    line_actions.append('+'.join(actions) if actions else 'line')

    if title_col is not None and title_col < len(row) and row[title_col] is not None:
        candidate = clean_title(row[title_col])
        if candidate:
            title_candidates.append(candidate)
    elif title_col is None:
        for value in row:
            if value is None or looks_like_number(value):
                continue
            candidate = clean_title(value)
            if candidate:
                title_candidates.append(candidate)
                break

if not wavelengths:
    raise ValueError(f'No spectral rows found in sheet {ws.title}')

wavelengths = np.array(wavelengths)
intensities = np.array(intensities, dtype=float)
line_width_factors = np.array(line_width_factors, dtype=float)

sheet_title = clean_title(ws.title)
is_generic_sheet_title = ws.title.strip().lower().startswith('sheet')
if sheet_title and not is_generic_sheet_title:
    spectrum_title = sheet_title
elif title_candidates:
    spectrum_title = title_candidates[0]
else:
    spectrum_title = os.path.splitext(excel_file)[0]

print(f'Selected newest Excel file: {excel_file}')
print(f'Loaded {len(wavelengths)} spectral lines from {excel_file} / {ws.title}')
if skipped_masked_lines:
    print(f'Skipped masked lines (`m`): {skipped_masked_lines}')

if raw_descriptor_counts:
    print('Raw descriptor strings found:', dict(raw_descriptor_counts))
if parsed_descriptor_counts:
    print('Parsed descriptor token counts:', dict(parsed_descriptor_counts))
if line_actions:
    print('Rendered action counts:', dict(Counter(line_actions)))
if unknown_descriptor_counts:
    print('Unknown descriptor tokens (check reference):', dict(unknown_descriptor_counts))

NameError: name 'wb' is not defined